# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}\n\n{metadata['description']}")

## 2. Data Overview
Review available record sets and fields, including their `@id`s.

We will programmatically list the available record sets, fields, and columns, referencing all by their `@id`.

In [ ]:
# Explore available record sets and their fields
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"  - Name: {rs.name}  |  @id: {rs.id}")
    print("    Fields and columns (with @id):")
    for field in rs.fields:
        print(f"      - Field name: {field.name}  |  @id: {field.id}  |  Data type: {field.data_type}")
        for col in getattr(field, 'columns', []):
            print(f"        - Column name: {getattr(col, 'name', None)} | @id: {getattr(col, 'id', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we demonstrate extracting all record sets into pandas DataFrames, referencing each by its `@id`. These can then be used for analysis.

In [ ]:
# Find all record set `@id`s
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Record Set IDs found:")
for rid in record_set_ids:
    print(f"  - {rid}")

# Load each record set as a DataFrame
dataframes = {}
for rid in record_set_ids:
    records = list(dataset.records(record_set=rid))
    df = pd.DataFrame(records)
    dataframes[rid] = df
    print(f"\nColumns in record set {rid}:")
    print(df.columns.tolist())
    print(f"First 3 rows in record set {rid}:")
    print(df.head(3))

# Choose a main record set for deeper analysis below:
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nMain record set selected for EDA: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**All fields referenced by their `@id`.**

In [ ]:
# For reproducibility, select a numeric field and group field by their @id from exploration above.

# If there are no record sets, cannot proceed (guard).
if not main_record_set_id:
    print("No record sets found in the dataset.")
else:
    df = dataframes[main_record_set_id]
    print(f"Analyzing DataFrame for record set @id: {main_record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    # Try to select a numeric field
    numeric_fields = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
    else:
        print("No numeric fields available for analysis.")
        numeric_field_id = None
    # Try to select a group field (categorical)
    if numeric_field_id:
        cat_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field_id = None
        # Choose a field that is not too unique.
        for f in cat_fields:
            unique_vals = df[f].nunique()
            if unique_vals > 1 and unique_vals < len(df) // 3:
                group_field_id = f
                break
        if group_field_id:
            print(f"Using group field: {group_field_id}")
    # Data filtering and normalization
    if numeric_field_id:
        threshold = df[numeric_field_id].quantile(0.5)  # median as threshold example
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold} (median): {len(filtered_df)} records")

        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped average of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset referencing the fields by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Show histogram and boxplot for selected numeric field
if main_record_set_id and numeric_field_id and not df[numeric_field_id].isnull().all():
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))

    sns.histplot(df[numeric_field_id].dropna(), ax=axs[0], kde=True)
    axs[0].set_title(f'Histogram for {numeric_field_id}')
    axs[0].set_xlabel(numeric_field_id)

    sns.boxplot(x=df[numeric_field_id].dropna(), ax=axs[1])
    axs[1].set_title(f'Boxplot for {numeric_field_id}')

    plt.tight_layout()
    plt.show()

# If group_field_id available, boxplot by group
if main_record_set_id and numeric_field_id and group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f'{numeric_field_id} distribution by {group_field_id}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using `mlcroissant`, we loaded a dataset and referenced all data elements by their Croissant `@id`.
- The record sets and their structure (fields/columns) can be explored programmatically.
- We extracted and performed simple EDA and visualizations, demonstrating best practices when using FAIR, schema-driven data.

**Next Steps:** Use these DataFrames for more domain-specific analyses, modeling, or further FAIR compliance assessments.